## **Learning Objectives**

By completing these exercises, you will:

- Understand Retrieval-Augmented Generation (RAG) and its components.
- Load, preprocess, and handle PDF documents effectively.
- Convert textual data into embeddings for efficient retrieval.
- Implement and test document retrieval systems using LangChain and FAISS.
- Integrate retrieval systems with free Language Models (LLMs) from ChatGroq .
- Build an interactive chat-based Q&A system.

---

## **Exercise 1: Setup and Warm-up**

In this exercise, you'll set up your environment and select a suitable language model.

**Steps:**

1. **Load Environment Variables:** Ensure your environment variables (e.g., API keys, tokens) are securely stored and loaded.
2. **Choose LLM:** Select a free LLM model from from ChatGroq. 
3. **Instantiate the Model:** Create an instance of your chosen model.


In [1]:
# 1.1 Setup and Model Initialization

# a. Import necessary libraries
import os
from dotenv import load_dotenv
from langchain_community.chat_models import ChatOllama

# b. Load environment variables from .env file
load_dotenv()

# c. Instantiate the Model (Llama 3.2 via Ollama - runs fully offline)
# We use the 3b version for a good balance between speed and quality
model = ChatOllama(model="llama3.2:3b")

# d. Quick test to confirm model is working
response = model.invoke("Say hello in one word")
print(f"Model response: {response.content}")

print("✅ 01 Step complete: Model is ready.")

/Users/asimeoa/aipm-1711/ds-rag-pipeline_sia/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/var/folders/_v/rjrhrzpx5l1bcyfw7cznsxsw0000gn/T/ipykernel_17982/2216707356.py:13: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the :class:`~langchain-ollama package and should be used instead. To use it run `pip install -U :class:`~langchain-ollama` and import as `from :class:`~langchain_ollama import ChatOllama``.
  model = ChatOllama(model="llama3.2:3b")


Model response: Hello.
✅ 01 Step complete: Model is ready.


---

## **Exercise 2: Data Ingestion**

In this exercise, you'll learn to load PDF data into a Python environment.

**Steps:**

1. **Import PDF Loader:** Use LangChain’s `PyPDFLoader`.
2. **Load PDF File:** Create a function to read the PDF file.
3. **Display PDF Content:** Print the number of pages and first page content.

In [2]:
# 2.1 Data Ingestion

# a. Import PyPDFLoader
from langchain_community.document_loaders import PyPDFLoader

# b. Define the function to load PDF
def load_pdf(pdf_path):
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()
    return documents

# c. Load your PDF and print status
# Adjust path if your PDF is in the 'documents' folder
file_path = "../documents/paracetamol.pdf"
documents = load_pdf(file_path)

if documents:
    print(f"✅ 02 Success! {len(documents)} pages loaded.")
    print(f"Preview: {documents[0].page_content[:150]}...")
else:
    print("❌ 02 Failed: No pages loaded. Check your file path.")

✅ 02 Success! 3 pages loaded.
Preview: 202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA
Printed Leaﬂet for  Paracetamo...


---

## **Exercise 3: Document Chunking**

This exercise introduces splitting large documents into manageable text chunks.

**Steps:**

1. **Import Text Splitter:** Use `RecursiveCharacterTextSplitter`.
2. **Chunk Document:** Write a function that splits loaded documents into chunks.
3. **Test Function:** Verify by displaying the resulting chunks.


In [3]:
# 3.1 Document Chunking

# a. Import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter

# b. Define the chunking function
def chunk_documents(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        add_start_index=True
    )
    chunks = text_splitter.split_documents(documents)
    return chunks

# c. Execute chunking and display results
chunks = chunk_documents(documents)
print(f"✅ 03 Created {len(chunks)} chunks from the PDF.")
print(f"Preview first chunk: {chunks[0].page_content[:150]}...")

✅ 03 Created 45 chunks from the PDF.
Preview first chunk: 202211
178 mm
422 mm
178 mm
422 mm
Front Side Back Side
 Paracetamol 500mg Tablets
178 x 422mm
178 x 30mm
358
202211
NA
Printed Leaﬂet for  Paracetamo...



---

## **Exercise 4: Embedding and Storage**

In this exercise, you will create embeddings from text chunks and store them efficiently.

**Steps:**

1. **Choose Embedding Model:** Use `sentence-transformers/all-mpnet-base-v2` from Hugging Face.
2. **Generate Embeddings:** Transform document chunks into embeddings.
3. **Store Embeddings:** Save these embeddings using FAISS locally.


In [4]:
# 4.1 Embedding and Storage (FAISS)

from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings

# HuggingFace runs directly in Python - no external process, more stable on M1
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Creating FAISS index... wait...")
vector_db_faiss = FAISS.from_documents(chunks, embeddings)

faiss_folder = "faiss_index_v2"
vector_db_faiss.save_local(faiss_folder)

print(f"✅ 04 Step complete: FAISS index saved in '{faiss_folder}'.")

Creating FAISS index... wait...
✅ 04 Step complete: FAISS index saved in 'faiss_index_v2'.


---

## **Exercise 5: Retrieval from FAISS**

Here, you will learn how to retrieve documents from a vector database using embeddings.

**Steps:**

1. **Load Embeddings:** Load stored embeddings from the FAISS database.
2. **Implement Retrieval:** Create logic to retrieve relevant chunks based on queries.
3. **Test Retriever:** Execute retrieval using sample queries.

In [5]:
# 5.1 Retrieval Testing
import time
from langchain_community.vectorstores import FAISS

print("🛠️ Starting Fresh Load with Safety Buffer...")

try:
    # b. Load the FAISS index - reuse existing embeddings object from cell 4.1
    print("📦 Loading index into RAM...")
    vector_db_faiss = FAISS.load_local(
        "faiss_index_v2",
        embeddings,  # <-- dasselbe Objekt wie in 4.1, kein neu laden!
        allow_dangerous_deserialization=True
    )

    # c. Define retrieval function
    def retrieve_docs(query):
        """Searches for relevant sections in the PDF index"""
        return vector_db_faiss.similarity_search(query, k=3)

    print("✅ 05.1 Step complete: retrieve_docs is ready.")

except Exception as e:
    print(f"❌ Safety Stop: {e}")

🛠️ Starting Fresh Load with Safety Buffer...
📦 Loading index into RAM...
✅ 05.1 Step complete: retrieve_docs is ready.


In [6]:
# 5.1.5 Quick Sanity Check before 5.2

print(f"Index loaded: {vector_db_faiss is not None}")
print(f"retrieve_docs defined: {callable(retrieve_docs)}")
print("✅ Ready for 5.2!")

Index loaded: True
retrieve_docs defined: True
✅ Ready for 5.2!


In [7]:
# 5.2 Retrieval Testing - Minimal Version

test_query = "What is paracetamol used for?"
print(f"🔍 Searching: '{test_query}'...")

try:
    # Direct call without function wrapper
    results = vector_db_faiss.similarity_search(test_query, k=2)
    print(f"✅ 05.2 Found {len(results)} results.")
    print(results[0].page_content[:200])

except Exception as e:
    print(f"❌ Error: {e}")

🔍 Searching: 'What is paracetamol used for?'...
✅ 05.2 Found 2 results.
hoher Anionenlücke), die bei einem Anstieg der Plasmasäure auftre-
ten, wenn Flucloxacillin gleichzeitig mit Paracetamol verwendet wird, 
in der Regel bei Vorliegen von Risikofaktoren (siehe 2. „Anwen


---

## **Exercise 6: Connecting Retrieval with LLM**

You'll now connect document retrieval with the Language Model.

**Steps:**

1. **Create Retrieval Chain:** Link your retrieval system to your instantiated LLM.
2. **Test the Chain:** Confirm it works by generating answers from retrieved documents.

In [8]:
# 6.1 Create the RAG Chain

from langchain.chains import RetrievalQA

try:
    print("🏗️ Connecting Model and Database...")
    
    # Connect FAISS retriever with the LLM
    qa_chain = RetrievalQA.from_chain_type(
        llm=model,
        chain_type="stuff",
        retriever=vector_db_faiss.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=False
    )
    
    print("✅ 06.1 Step complete: qa_chain is ready.")

except Exception as e:
    print(f"❌ Error during 06.1: {e}")

🏗️ Connecting Model and Database...
✅ 06.1 Step complete: qa_chain is ready.


In [9]:
# 6.2 Execution and Testing

import time

# Define your question

question = "Wofür wird Paracetamol laut Dokument verwendet?"

try:

    print(f"🤖 AI is reading the PDF and generating an answer for: '{question}'")

    

    # Safety: Give the system a short breath

    time.sleep(1)

    

    # Run the chain

    start_time = time.time()

    response = qa_chain.invoke(question)

    end_time = time.time()

    

    print(f"✅ 6.2 Calculation finished in {round(end_time - start_time, 2)} seconds.\n")

    print("--- AI ANSWER ---")

    print(response["result"])

    

except Exception as e:

    print(f"❌ Error during execution: {e}")

🤖 AI is reading the PDF and generating an answer for: 'Wofür wird Paracetamol laut Dokument verwendet?'
✅ 6.2 Calculation finished in 4.6 seconds.

--- AI ANSWER ---
Laut dem Dokument wird Paracetamol als schmerzstillendes und fiebersenkendes Arzneimittel (Analgetikum und Antipyretikum) eingesetzt, um leichte bis mäßige Schmerzen und Fieber zu behandeln.


---

## **Exercise 7: Interactive Chat System**

In the final exercise, build an interactive chat-based query system.

**Steps:**

1. **Create Chat Interface:** Develop a simple function for interactive querying.
2. **Run the Chat:** Allow users to ask questions and receive immediate responses.


In [10]:
# 7.1 Define the interactive chat function

def chat(question):
    """Sends a question to the RAG system and returns the answer"""
    try:
        response = qa_chain.invoke(question)
        return response["result"]
    except Exception as e:
        return f"❌ Error: {e}"

print("✅ 07.1 Step complete: chat function is ready.")

✅ 07.1 Step complete: chat function is ready.


In [17]:
# 7.2 Run the interactive chat system

print("💬 RAG Chat System - Test Run\n")

# Add your test questions here
test_questions = [
    "was ist der unterschied zu aspirin? ?",
    "Wofür wird Paracetamol verwendet?",
    "wo in dem text wird Flucloxacillin beschreiben?"
]

for question in test_questions:
    print(f"You: {question}")
    answer = chat(question)
    print(f"🤖 AI: {answer}\n")
    print("-" * 40 + "\n")

💬 RAG Chat System - Test Run

You: was ist der unterschied zu aspirin? ?
🤖 AI: Ich weiß nicht, was das genaue Unterschied zwischen Paracetamol und Aspirin ist. Es scheint, dass Paracetamol ein anderes Wirkstoff als Aspirin ist, aber ich kann keine spezifischen Informationen darüber liefern. Wenn du mehr über die chemische Struktur oder die Wirkungsweise der beiden Medikamente wissen möchtest, kann ich versuchen, das zu recherchieren...

----------------------------------------

You: Wofür wird Paracetamol verwendet?
🤖 AI: Paracetamol wird zur symptomatischen Behandlung von leichten bis mittelschweren Schmerzen und Fieber eingesetzt. Es ist ein schmerzstillendes, fiebersenkendes Arzneimittel (Analgetikum und Antipyretikum).

----------------------------------------

You: wo in dem text wird Flucloxacillin beschreiben?
🤖 AI: Flucloxacillin wird im Text im Kontext einer möglichen Kombination mit Paracetamol beschrieben, insbesondere bei bestimmten Risikogruppen wie Patienten mit schwerer 

---

## **Conclusion & Reflection**

After completing these exercises:

- Summarize key concepts learned.
- Reflect on the effectiveness and limitations of the free LLM and RAG system you've built.
- Consider how you might improve or extend your system in practical applications.

---